### AI LAB 6 TASK-1

In [1]:
import random

In [2]:
N = 7
POPULATION_SIZE = 100
MAX_GENERATIONS = 1000
MUTATION_RATE = 0.1
TOURNAMENT_SIZE = 3
MAX_FITNESS = N * (N - 1) // 2   # 21

In [3]:
# PROCESS 1: Chromosome Representation (Random Permutation)
def create_chromosome():
    return random.sample(range(N), N)

In [4]:
# PROCESS 2: Fitness Function (Count Non-Attacking Pairs)
def fitness(chromosome):
    non_attacking = 0
    for i in range(N):
        for j in range(i + 1, N):
            if abs(chromosome[i] - chromosome[j]) != abs(i - j):
                non_attacking += 1
    return non_attacking

In [5]:
# PROCESS 3: Selection (Tournament Selection)
def tournament_selection(population):
    selected = []
    for _ in range(2):  # select two parents
        tournament = random.sample(population, TOURNAMENT_SIZE)
        best = max(tournament, key=fitness)
        selected.append(best)
    return selected[0], selected[1]

In [6]:
# PROCESS 4: Crossover (Order Crossover - OX)
def order_crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [None] * size

    child[start:end+1] = parent1[start:end+1]

    current = 0
    for gene in parent2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child

In [7]:
# PROCESS 5: Mutation (Swap Mutation)
def swap_mutation(chromosome):
    idx1, idx2 = random.sample(range(N), 2)
    chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome

In [8]:
# PROCESS 6: TERMINATION CONDITION & Genetic Algorithm Main Loop 
def genetic_algorithm():

    population = [create_chromosome() for _ in range(POPULATION_SIZE)]
    best_solution = None
    best_fitness = -1

    for generation in range(MAX_GENERATIONS):

        fitness_scores = [fitness(ind) for ind in population]
        current_best = max(fitness_scores)
        if current_best > best_fitness:
            best_fitness = current_best
            best_solution = population[fitness_scores.index(current_best)]

        # TERMINATION CONDITION: Check if we found a perfect solution
        if best_fitness == MAX_FITNESS:
            print(f"Perfect solution found at generation {generation+1}")
            break

        new_population = []
        while len(new_population) < POPULATION_SIZE:
            parent1, parent2 = tournament_selection(population)
            child = order_crossover(parent1, parent2)
            if random.random() < MUTATION_RATE:
                child = swap_mutation(child)
            new_population.append(child)

        population = new_population

    return best_solution, best_fitness

In [9]:
# SOLUTION: Run GA and Display Result
solution, fit = genetic_algorithm()
print(f"Camera Placement: {solution}")
print(f"Fitness: {fit}")

Perfect solution found at generation 3
Camera Placement: [2, 4, 6, 1, 3, 5, 0]
Fitness: 21


### AI LAB 6 TASK-2

In [10]:
import random

chromosomes = ['C1', 'C2', 'C3', 'C4', 'C5']
fitnesses = [12, 20, 15, 8, 25]

In [11]:
# PROCESS 1: Roulette Wheel Selection
# DESCRIPTION:
# Each individual gets a slice of a roulette wheel proportional to its fitness.
# A random number selects the winner.
def roulette_wheel_selection(chromosomes, fitnesses):
    total = sum(fitnesses)
    pick = random.uniform(0, total)
    cumulative = 0
    for ch, fit in zip(chromosomes, fitnesses):
        cumulative += fit
        if cumulative > pick:
            return ch
    return chromosomes[-1]

In [12]:
# PROCESS 2: Tournament Selection
# DESCRIPTION:
# Randomly pick k individuals, then return the one with the highest fitness.
def tournament_selection(chromosomes, fitnesses, k=3):
    participants = random.sample(list(zip(chromosomes, fitnesses)), k)
    winner = max(participants, key=lambda x: x[1])[0]
    return winner

In [13]:
# PROCESS 3: Rank Selection
# DESCRIPTION:
# Sort by fitness, assign ranks (1 = worst, N = best).
# Probability is proportional to rank (higher rank → higher chance).
def rank_selection(chromosomes, fitnesses):

    paired = sorted(zip(chromosomes, fitnesses), key=lambda x: x[1])
    n = len(paired)
    ranks = list(range(1, n+1))
    total_rank = sum(ranks)
    probs = [r / total_rank for r in ranks]

    pick = random.uniform(0, 1)
    cumulative = 0
    for (ch, _), p in zip(paired, probs):
        cumulative += p
        if cumulative > pick:
            return ch
    return paired[-1][0]

In [14]:
# LAST STEP: Test each selection technique multiple times
print("=== Roulette Wheel Selection (5 runs) ===")
for _ in range(5):
    print(roulette_wheel_selection(chromosomes, fitnesses))

print("\n=== Tournament Selection (5 runs) ===")
for _ in range(5):
    print(tournament_selection(chromosomes, fitnesses))

print("\n=== Rank Selection (5 runs) ===")
for _ in range(5):
    print(rank_selection(chromosomes, fitnesses))

=== Roulette Wheel Selection (5 runs) ===
C2
C4
C3
C4
C3

=== Tournament Selection (5 runs) ===
C5
C5
C3
C2
C2

=== Rank Selection (5 runs) ===
C1
C5
C5
C1
C3


### AI LAB 6 TASK-3

In [15]:
import random

parent1 = [1, 3, 5, 0, 6, 4, 2]
parent2 = [2, 5, 1, 6, 0, 3, 4]

In [16]:
# PROCESS 1: One-Point Crossover (with duplicate repair)
def one_point_crossover(p1, p2):
    n = len(p1)
    point = random.randint(1, n - 1)
    child = p1[:point] + p2[point:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child

In [17]:
# PROCESS 2: Two-Point Crossover (with duplicate repair)
def two_point_crossover(p1, p2):
    n = len(p1)
    points = sorted(random.sample(range(1, n), 2))
    child = p1[:points[0]] + p2[points[0]:points[1]] + p1[points[1]:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child

In [18]:
# PROCESS 3: Order Crossover (OX)
def order_crossover(p1, p2):
    n = len(p1)
    start, end = sorted(random.sample(range(n), 2))
    child = [None] * n

    child[start:end+1] = p1[start:end+1]

    current = 0
    for gene in p2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child

In [19]:
# LAST STEP: Test each crossover technique
print("One-Point Crossover:", one_point_crossover(parent1, parent2))
print("Two-Point Crossover:", two_point_crossover(parent1, parent2))
print("Order Crossover:", order_crossover(parent1, parent2))

One-Point Crossover: [1, 3, 5, 0, 6, 2, 4]
Two-Point Crossover: [1, 5, 0, 6, 3, 4, 2]
Order Crossover: [1, 3, 5, 0, 2, 6, 4]


### AI LAB 6 TASK-4

In [20]:
import random

chromosome = [1, 3, 5, 0, 6, 4, 2]

In [21]:
# PROCESS 1: Swap Mutation
def swap_mutation(chromo):
    idx1, idx2 = random.sample(range(len(chromo)), 2)
    chromo[idx1], chromo[idx2] = chromo[idx2], chromo[idx1]
    return chromo

In [22]:
# PROCESS 2: Scramble Mutation
def scramble_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    segment = chromo[start:end+1]
    random.shuffle(segment)
    chromo[start:end+1] = segment
    return chromo

In [23]:
# PROCESS 3: Inversion Mutation
def inversion_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    chromo[start:end+1] = reversed(chromo[start:end+1])
    return chromo

In [24]:
# LAST STEP: Test each mutation technique
print("Swap Mutation:", swap_mutation(chromosome.copy()))
print("Scramble Mutation:", scramble_mutation(chromosome.copy()))
print("Inversion Mutation:", inversion_mutation(chromosome.copy()))

Swap Mutation: [1, 3, 6, 0, 5, 4, 2]
Scramble Mutation: [1, 3, 5, 0, 4, 6, 2]
Inversion Mutation: [5, 3, 1, 0, 6, 4, 2]
